# RSQR repo import and validation notebook

This notebook is meant to run from a local clone or from Colab after the repo is mounted or cloned. The goal is to make the repo importable, then run the real project tests from the notebook itself.


In [ ]:
# Install the minimum dependencies needed for the project tests and model smoke run.
# In Colab, this is safe to run from a fresh runtime.
!python -V
!pip install -q --upgrade pip
!pip install -q transformers accelerate sentencepiece torch

import os
import sys
from pathlib import Path

# Try a few common repo locations. This makes the notebook portable between local VS Code,
# Jupyter, and Colab.
candidates = [
    Path('/content/kv-eviction'),
    Path('/workspace/kv-eviction'),
    Path.cwd() / 'kv-eviction',
    Path.cwd(),
]

repo_root = None
for candidate in candidates:
    if candidate.exists() and (candidate / 'src').exists():
        repo_root = candidate
        break

if repo_root is None:
    # Last resort: find the first folder that contains a src directory under the current tree.
    possible = sorted(Path.cwd().glob('**/src'), key=lambda p: len(p.parts))
    if possible:
        repo_root = possible[0].parent

if repo_root is None:
    raise FileNotFoundError(
        'Could not locate the repo. Clone or mount it to /content/kv-eviction or /workspace/kv-eviction '
        'or run this notebook from inside the repo.'
    )

sys.path.insert(0, str(repo_root))
print('repo_root =', repo_root)

from src.eviction import EvictionManager, WindowState
from src.index_map import IndexMap
from src.rope import apply_rope, precompute_rope_freqs
from src.shadow_cache import ShadowCache

print('imports_ok = True')

In [ ]:
# Run the real project tests from within the notebook.
# This is the easiest way to validate that the repo imports correctly and the package is working.

import pytest

result = pytest.main([
    '-q',
    'tests/test_eviction_correctness.py',
    'tests/test_policy_cache_generation.py',
])
print('pytest_exit_code =', result)

In [ ]:
# Small local smoke check using the real package primitives.
import torch

freqs = precompute_rope_freqs(1024, 8, device=torch.device('cpu'))
raw_key = torch.randn(8, dtype=torch.float32)
shadow_cache = ShadowCache(survivor_every=8)
shadow_cache.add(token_id=42, raw_key=raw_key, rotated_key=raw_key.clone(), is_survivor=True)
index_map = IndexMap()
index_map.compact([42])
manager = EvictionManager(freqs)
boundary = manager.on_boundary(shadow_cache, index_map, WindowState(window_size=64, evict_n=8))
rotated = boundary['rotated'][0]['key']
expected = apply_rope(
    raw_key.unsqueeze(0).unsqueeze(0),
    torch.tensor([0.0], dtype=torch.float32),
    freqs,
).squeeze(0).squeeze(0)
max_abs_diff = (rotated - expected).abs().max().item()
print('smoke_max_abs_diff =', max_abs_diff)
assert max_abs_diff < 1e-5, max_abs_diff

## Notes

- This notebook is the repo-import bridge.
- It is not meant to be a fancy benchmark runner.
- The goal is to let you import the package and run the real project tests directly in the notebook environment.
